# Integer Programming in Pure Python
## Carpenter Problem: Exact Methods for Small Integer Models

This notebook presents five exact approaches for small integer optimization problems in pure Python:

1. Naive enumeration
2. Backtracking
3. Optimized enumeration
4. Dynamic Programming
5. Branch and Bound

We will solve:
- the original 2-product problem
- the extended 3-product problem

Each method reports:
- the best solution found
- the profit
- the execution time

In [1]:
from __future__ import annotations

from functools import wraps, lru_cache
from itertools import combinations
from math import ceil, floor, isclose
from time import perf_counter

In [2]:
EPSILON = 1e-9


def timed(func):
    """Decorator to measure execution time."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

In [3]:
def dot_product(vector_a, vector_b):
    return sum(a * b for a, b in zip(vector_a, vector_b))


def is_integer_value(value: float) -> bool:
    return isclose(value, round(value), abs_tol=EPSILON)


def solve_linear_system(matrix, rhs):
    """
    Solve A x = b using Gaussian elimination with partial pivoting.
    Returns a list with the solution, or None if the system is singular.
    """
    n = len(rhs)
    augmented = [row[:] + [rhs_value] for row, rhs_value in zip(matrix, rhs)]

    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(augmented[r][col]))
        if abs(augmented[pivot_row][col]) < EPSILON:
            return None

        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]

        pivot = augmented[col][col]
        for j in range(col, n + 1):
            augmented[col][j] /= pivot

        for row in range(n):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, n + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[i][n] for i in range(n)]


def solve_lp_relaxation_by_vertices(objective, constraints, bounds):
    """
    Solve a small LP relaxation by enumerating vertices.

    Maximize:
        c^T x

    Subject to:
        A x <= b
        lower_bound_i <= x_i <= upper_bound_i
    """
    number_of_variables = len(objective)
    candidate_equalities = []

    for coefficients, rhs in constraints:
        candidate_equalities.append((coefficients[:], rhs))

    for i, (lower_bound, upper_bound) in enumerate(bounds):
        unit_vector = [0.0] * number_of_variables
        unit_vector[i] = 1.0
        candidate_equalities.append((unit_vector[:], lower_bound))
        candidate_equalities.append((unit_vector[:], upper_bound))

    best_point = None
    best_value = float("-inf")

    for selected_equalities in combinations(candidate_equalities, number_of_variables):
        matrix = [list(eq[0]) for eq in selected_equalities]
        rhs = [eq[1] for eq in selected_equalities]

        solution = solve_linear_system(matrix, rhs)
        if solution is None:
            continue

        feasible = True

        for value, (lower_bound, upper_bound) in zip(solution, bounds):
            if value < lower_bound - EPSILON or value > upper_bound + EPSILON:
                feasible = False
                break

        if not feasible:
            continue

        for coefficients, rhs_value in constraints:
            if dot_product(coefficients, solution) > rhs_value + EPSILON:
                feasible = False
                break

        if not feasible:
            continue

        objective_value = dot_product(objective, solution)
        if objective_value > best_value + EPSILON:
            best_value = objective_value
            best_point = solution

    if best_point is None:
        return None, None

    return best_point, best_value

In [4]:
def branch_and_bound(objective, constraints, root_bounds, variable_names):
    """
    Generic Branch and Bound for small integer linear programs.
    LP relaxations are solved with vertex enumeration.
    """
    best_integer_value = float("-inf")
    best_integer_solution = None

    stack = [root_bounds]

    while stack:
        bounds = stack.pop()

        lp_solution, lp_value = solve_lp_relaxation_by_vertices(
            objective=objective,
            constraints=constraints,
            bounds=bounds,
        )

        if lp_solution is None:
            continue

        if lp_value <= best_integer_value + EPSILON:
            continue

        fractional_index = None
        for i, value in enumerate(lp_solution):
            if not is_integer_value(value):
                fractional_index = i
                break

        if fractional_index is None:
            integer_solution = [int(round(x)) for x in lp_solution]
            integer_value = dot_product(objective, integer_solution)

            if integer_value > best_integer_value + EPSILON:
                best_integer_value = integer_value
                best_integer_solution = {
                    name: value
                    for name, value in zip(variable_names, integer_solution)
                }
                best_integer_solution["profit"] = int(round(integer_value))
            continue

        fractional_value = lp_solution[fractional_index]
        lower_branch_upper_bound = floor(fractional_value)
        upper_branch_lower_bound = ceil(fractional_value)

        left_bounds = [tuple(bound) for bound in bounds]
        left_lb, left_ub = left_bounds[fractional_index]
        left_bounds[fractional_index] = (
            left_lb,
            min(left_ub, float(lower_branch_upper_bound)),
        )
        if left_bounds[fractional_index][0] <= left_bounds[fractional_index][1] + EPSILON:
            stack.append(left_bounds)

        right_bounds = [tuple(bound) for bound in bounds]
        right_lb, right_ub = right_bounds[fractional_index]
        right_bounds[fractional_index] = (
            max(right_lb, float(upper_branch_lower_bound)),
            right_ub,
        )
        if right_bounds[fractional_index][0] <= right_bounds[fractional_index][1] + EPSILON:
            stack.append(right_bounds)

    return best_integer_solution

## Problem 1: Two Products

Variables:
- rockers
- stools

Objective:
- maximize profit

Resources:
- small parts = 54
- medium parts = 30
- large parts = 30

In [5]:
# Available resources
SMALL_PARTS = 54
MEDIUM_PARTS = 30
LARGE_PARTS = 30

# Product 1: Rocking chair
ROCKER_SMALL = 18
ROCKER_MEDIUM = 9
ROCKER_LARGE = 2
ROCKER_PROFIT = 70

# Product 2: Stool
STOOL_SMALL = 2
STOOL_MEDIUM = 2
STOOL_LARGE = 1
STOOL_PROFIT = 15

TWO_PRODUCT_OBJECTIVE = [ROCKER_PROFIT, STOOL_PROFIT]
TWO_PRODUCT_VARIABLE_NAMES = ["rockers", "stools"]

TWO_PRODUCT_CONSTRAINTS = [
    ([ROCKER_SMALL, STOOL_SMALL], SMALL_PARTS),
    ([ROCKER_MEDIUM, STOOL_MEDIUM], MEDIUM_PARTS),
    ([ROCKER_LARGE, STOOL_LARGE], LARGE_PARTS),
]

MAX_ROCKERS_2 = min(
    SMALL_PARTS // ROCKER_SMALL,
    MEDIUM_PARTS // ROCKER_MEDIUM,
    LARGE_PARTS // ROCKER_LARGE,
)

MAX_STOOLS_2 = min(
    SMALL_PARTS // STOOL_SMALL,
    MEDIUM_PARTS // STOOL_MEDIUM,
    LARGE_PARTS // STOOL_LARGE,
)

TWO_PRODUCT_ROOT_BOUNDS = [
    (0.0, float(MAX_ROCKERS_2)),
    (0.0, float(MAX_STOOLS_2)),
]

In [6]:
@timed
def solve_two_products_naive():
    best_profit = -1
    best_solution = None

    for rockers in range(MAX_ROCKERS_2 + 1):
        for stools in range(MAX_STOOLS_2 + 1):
            if (
                ROCKER_SMALL * rockers + STOOL_SMALL * stools <= SMALL_PARTS
                and ROCKER_MEDIUM * rockers + STOOL_MEDIUM * stools <= MEDIUM_PARTS
                and ROCKER_LARGE * rockers + STOOL_LARGE * stools <= LARGE_PARTS
            ):
                profit = ROCKER_PROFIT * rockers + STOOL_PROFIT * stools

                if profit > best_profit:
                    best_profit = profit
                    best_solution = {
                        "rockers": rockers,
                        "stools": stools,
                        "profit": profit,
                    }

    return best_solution

In [7]:
@timed
def solve_two_products_backtracking():
    best_profit = -1
    best_solution = None
    max_values = [MAX_ROCKERS_2, MAX_STOOLS_2]

    def backtrack(index, current_solution):
        nonlocal best_profit, best_solution

        rockers, stools = current_solution

        used_small = ROCKER_SMALL * rockers + STOOL_SMALL * stools
        used_medium = ROCKER_MEDIUM * rockers + STOOL_MEDIUM * stools
        used_large = ROCKER_LARGE * rockers + STOOL_LARGE * stools

        if used_small > SMALL_PARTS or used_medium > MEDIUM_PARTS or used_large > LARGE_PARTS:
            return

        if index == 2:
            profit = ROCKER_PROFIT * rockers + STOOL_PROFIT * stools
            if profit > best_profit:
                best_profit = profit
                best_solution = {
                    "rockers": rockers,
                    "stools": stools,
                    "profit": profit,
                }
            return

        for value in range(max_values[index] + 1):
            current_solution[index] = value
            backtrack(index + 1, current_solution)

        current_solution[index] = 0

    backtrack(0, [0, 0])
    return best_solution

In [8]:
@timed
def solve_two_products_optimized():
    best_profit = -1
    best_solution = None

    for rockers in range(MAX_ROCKERS_2 + 1):
        remaining_small = SMALL_PARTS - ROCKER_SMALL * rockers
        remaining_medium = MEDIUM_PARTS - ROCKER_MEDIUM * rockers
        remaining_large = LARGE_PARTS - ROCKER_LARGE * rockers

        if remaining_small < 0 or remaining_medium < 0 or remaining_large < 0:
            continue

        max_stools = min(
            remaining_small // STOOL_SMALL,
            remaining_medium // STOOL_MEDIUM,
            remaining_large // STOOL_LARGE,
        )

        profit = ROCKER_PROFIT * rockers + STOOL_PROFIT * max_stools

        if profit > best_profit:
            best_profit = profit
            best_solution = {
                "rockers": rockers,
                "stools": int(max_stools),
                "profit": int(profit),
            }

    return best_solution

In [9]:
@timed
def solve_two_products_dp():
    @lru_cache(maxsize=None)
    def dp(remaining_small, remaining_medium, remaining_large, index):
        if index == 2:
            return 0, (0, 0)

        if index == 0:
            max_units = min(
                remaining_small // ROCKER_SMALL,
                remaining_medium // ROCKER_MEDIUM,
                remaining_large // ROCKER_LARGE,
            )
            unit_small, unit_medium, unit_large, unit_profit = (
                ROCKER_SMALL, ROCKER_MEDIUM, ROCKER_LARGE, ROCKER_PROFIT
            )
        else:
            max_units = min(
                remaining_small // STOOL_SMALL,
                remaining_medium // STOOL_MEDIUM,
                remaining_large // STOOL_LARGE,
            )
            unit_small, unit_medium, unit_large, unit_profit = (
                STOOL_SMALL, STOOL_MEDIUM, STOOL_LARGE, STOOL_PROFIT
            )

        best_profit = -1
        best_counts = None

        for units in range(max_units + 1):
            next_profit, next_counts = dp(
                remaining_small - units * unit_small,
                remaining_medium - units * unit_medium,
                remaining_large - units * unit_large,
                index + 1,
            )

            total_profit = units * unit_profit + next_profit

            if total_profit > best_profit:
                best_profit = total_profit
                if index == 0:
                    best_counts = (units, next_counts[1])
                else:
                    best_counts = (next_counts[0], units)

        return best_profit, best_counts

    profit, counts = dp(SMALL_PARTS, MEDIUM_PARTS, LARGE_PARTS, 0)
    return {
        "rockers": counts[0],
        "stools": counts[1],
        "profit": profit,
    }

In [10]:
@timed
def solve_two_products_branch_and_bound():
    return branch_and_bound(
        objective=TWO_PRODUCT_OBJECTIVE,
        constraints=TWO_PRODUCT_CONSTRAINTS,
        root_bounds=TWO_PRODUCT_ROOT_BOUNDS,
        variable_names=TWO_PRODUCT_VARIABLE_NAMES,
    )

In [11]:
two_naive_result, two_naive_time = solve_two_products_naive()
two_backtracking_result, two_backtracking_time = solve_two_products_backtracking()
two_optimized_result, two_optimized_time = solve_two_products_optimized()
two_dp_result, two_dp_time = solve_two_products_dp()
two_bb_result, two_bb_time = solve_two_products_branch_and_bound()

print("=== TWO-PRODUCT RESULTS ===")
print(f"Naive             -> {two_naive_result}, time = {two_naive_time:.8f} seconds")
print(f"Backtracking      -> {two_backtracking_result}, time = {two_backtracking_time:.8f} seconds")
print(f"Optimized         -> {two_optimized_result}, time = {two_optimized_time:.8f} seconds")
print(f"Dynamic Programming -> {two_dp_result}, time = {two_dp_time:.8f} seconds")
print(f"Branch and Bound  -> {two_bb_result}, time = {two_bb_time:.8f} seconds")

=== TWO-PRODUCT RESULTS ===
Naive             -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00003125 seconds
Backtracking      -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00004637 seconds
Optimized         -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00000671 seconds
Dynamic Programming -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00004271 seconds
Branch and Bound  -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00061012 seconds


## Problem 2: Three Products

Variables:
- rockers
- stools
- new_items

Objective:
- maximize profit

Resources:
- small parts = 54
- medium parts = 30
- large parts = 30

In [12]:
# Product 3: New item
NEW_ITEM_SMALL = 4
NEW_ITEM_MEDIUM = 4
NEW_ITEM_LARGE = 0
NEW_ITEM_PROFIT = 45

THREE_PRODUCT_OBJECTIVE = [ROCKER_PROFIT, STOOL_PROFIT, NEW_ITEM_PROFIT]
THREE_PRODUCT_VARIABLE_NAMES = ["rockers", "stools", "new_items"]

THREE_PRODUCT_CONSTRAINTS = [
    ([ROCKER_SMALL, STOOL_SMALL, NEW_ITEM_SMALL], SMALL_PARTS),
    ([ROCKER_MEDIUM, STOOL_MEDIUM, NEW_ITEM_MEDIUM], MEDIUM_PARTS),
    ([ROCKER_LARGE, STOOL_LARGE, NEW_ITEM_LARGE], LARGE_PARTS),
]

MAX_ROCKERS_3 = min(
    SMALL_PARTS // ROCKER_SMALL,
    MEDIUM_PARTS // ROCKER_MEDIUM,
    LARGE_PARTS // ROCKER_LARGE,
)

MAX_STOOLS_3 = min(
    SMALL_PARTS // STOOL_SMALL,
    MEDIUM_PARTS // STOOL_MEDIUM,
    LARGE_PARTS // STOOL_LARGE,
)

MAX_NEW_ITEMS_3 = min(
    SMALL_PARTS // NEW_ITEM_SMALL,
    MEDIUM_PARTS // NEW_ITEM_MEDIUM,
)

THREE_PRODUCT_ROOT_BOUNDS = [
    (0.0, float(MAX_ROCKERS_3)),
    (0.0, float(MAX_STOOLS_3)),
    (0.0, float(MAX_NEW_ITEMS_3)),
]

In [13]:
@timed
def solve_three_products_naive():
    best_profit = -1
    best_solution = None

    for rockers in range(MAX_ROCKERS_3 + 1):
        for stools in range(MAX_STOOLS_3 + 1):
            for new_items in range(MAX_NEW_ITEMS_3 + 1):
                if (
                    ROCKER_SMALL * rockers + STOOL_SMALL * stools + NEW_ITEM_SMALL * new_items <= SMALL_PARTS
                    and ROCKER_MEDIUM * rockers + STOOL_MEDIUM * stools + NEW_ITEM_MEDIUM * new_items <= MEDIUM_PARTS
                    and ROCKER_LARGE * rockers + STOOL_LARGE * stools + NEW_ITEM_LARGE * new_items <= LARGE_PARTS
                ):
                    profit = (
                        ROCKER_PROFIT * rockers
                        + STOOL_PROFIT * stools
                        + NEW_ITEM_PROFIT * new_items
                    )

                    if profit > best_profit:
                        best_profit = profit
                        best_solution = {
                            "rockers": rockers,
                            "stools": stools,
                            "new_items": new_items,
                            "profit": profit,
                        }

    return best_solution

In [14]:
@timed
def solve_three_products_backtracking():
    best_profit = -1
    best_solution = None
    max_values = [MAX_ROCKERS_3, MAX_STOOLS_3, MAX_NEW_ITEMS_3]

    def backtrack(index, current_solution):
        nonlocal best_profit, best_solution

        rockers, stools, new_items = current_solution

        used_small = (
            ROCKER_SMALL * rockers +
            STOOL_SMALL * stools +
            NEW_ITEM_SMALL * new_items
        )
        used_medium = (
            ROCKER_MEDIUM * rockers +
            STOOL_MEDIUM * stools +
            NEW_ITEM_MEDIUM * new_items
        )
        used_large = (
            ROCKER_LARGE * rockers +
            STOOL_LARGE * stools +
            NEW_ITEM_LARGE * new_items
        )

        if used_small > SMALL_PARTS or used_medium > MEDIUM_PARTS or used_large > LARGE_PARTS:
            return

        if index == 3:
            profit = (
                ROCKER_PROFIT * rockers +
                STOOL_PROFIT * stools +
                NEW_ITEM_PROFIT * new_items
            )
            if profit > best_profit:
                best_profit = profit
                best_solution = {
                    "rockers": rockers,
                    "stools": stools,
                    "new_items": new_items,
                    "profit": profit,
                }
            return

        for value in range(max_values[index] + 1):
            current_solution[index] = value
            backtrack(index + 1, current_solution)

        current_solution[index] = 0

    backtrack(0, [0, 0, 0])
    return best_solution

In [15]:
@timed
def solve_three_products_optimized():
    best_profit = -1
    best_solution = None

    for rockers in range(MAX_ROCKERS_3 + 1):
        for new_items in range(MAX_NEW_ITEMS_3 + 1):
            remaining_small = SMALL_PARTS - ROCKER_SMALL * rockers - NEW_ITEM_SMALL * new_items
            remaining_medium = MEDIUM_PARTS - ROCKER_MEDIUM * rockers - NEW_ITEM_MEDIUM * new_items
            remaining_large = LARGE_PARTS - ROCKER_LARGE * rockers - NEW_ITEM_LARGE * new_items

            if remaining_small < 0 or remaining_medium < 0 or remaining_large < 0:
                continue

            max_stools = min(
                remaining_small // STOOL_SMALL,
                remaining_medium // STOOL_MEDIUM,
                remaining_large // STOOL_LARGE,
            )

            profit = (
                ROCKER_PROFIT * rockers
                + STOOL_PROFIT * max_stools
                + NEW_ITEM_PROFIT * new_items
            )

            if profit > best_profit:
                best_profit = profit
                best_solution = {
                    "rockers": rockers,
                    "stools": int(max_stools),
                    "new_items": int(new_items),
                    "profit": int(profit),
                }

    return best_solution

In [16]:
@timed
def solve_three_products_dp():
    @lru_cache(maxsize=None)
    def dp(remaining_small, remaining_medium, remaining_large, index):
        if index == 3:
            return 0, (0, 0, 0)

        if index == 0:
            max_units = min(
                remaining_small // ROCKER_SMALL,
                remaining_medium // ROCKER_MEDIUM,
                remaining_large // ROCKER_LARGE,
            )
            unit_small, unit_medium, unit_large, unit_profit = (
                ROCKER_SMALL, ROCKER_MEDIUM, ROCKER_LARGE, ROCKER_PROFIT
            )
        elif index == 1:
            max_units = min(
                remaining_small // STOOL_SMALL,
                remaining_medium // STOOL_MEDIUM,
                remaining_large // STOOL_LARGE,
            )
            unit_small, unit_medium, unit_large, unit_profit = (
                STOOL_SMALL, STOOL_MEDIUM, STOOL_LARGE, STOOL_PROFIT
            )
        else:
            max_units = min(
                remaining_small // NEW_ITEM_SMALL,
                remaining_medium // NEW_ITEM_MEDIUM,
                remaining_large // 1 if NEW_ITEM_LARGE > 0 else 10**9,
            )
            unit_small, unit_medium, unit_large, unit_profit = (
                NEW_ITEM_SMALL, NEW_ITEM_MEDIUM, NEW_ITEM_LARGE, NEW_ITEM_PROFIT
            )

        best_profit = -1
        best_counts = None

        for units in range(max_units + 1):
            next_profit, next_counts = dp(
                remaining_small - units * unit_small,
                remaining_medium - units * unit_medium,
                remaining_large - units * unit_large,
                index + 1,
            )

            total_profit = units * unit_profit + next_profit

            if total_profit > best_profit:
                best_profit = total_profit
                if index == 0:
                    best_counts = (units, next_counts[1], next_counts[2])
                elif index == 1:
                    best_counts = (next_counts[0], units, next_counts[2])
                else:
                    best_counts = (next_counts[0], next_counts[1], units)

        return best_profit, best_counts

    profit, counts = dp(SMALL_PARTS, MEDIUM_PARTS, LARGE_PARTS, 0)
    return {
        "rockers": counts[0],
        "stools": counts[1],
        "new_items": counts[2],
        "profit": profit,
    }

In [17]:
@timed
def solve_three_products_branch_and_bound():
    return branch_and_bound(
        objective=THREE_PRODUCT_OBJECTIVE,
        constraints=THREE_PRODUCT_CONSTRAINTS,
        root_bounds=THREE_PRODUCT_ROOT_BOUNDS,
        variable_names=THREE_PRODUCT_VARIABLE_NAMES,
    )

In [18]:
three_naive_result, three_naive_time = solve_three_products_naive()
three_backtracking_result, three_backtracking_time = solve_three_products_backtracking()
three_optimized_result, three_optimized_time = solve_three_products_optimized()
three_dp_result, three_dp_time = solve_three_products_dp()
three_bb_result, three_bb_time = solve_three_products_branch_and_bound()

print("=== THREE-PRODUCT RESULTS ===")
print(f"Naive               -> {three_naive_result}, time = {three_naive_time:.8f} seconds")
print(f"Backtracking        -> {three_backtracking_result}, time = {three_backtracking_time:.8f} seconds")
print(f"Optimized           -> {three_optimized_result}, time = {three_optimized_time:.8f} seconds")
print(f"Dynamic Programming -> {three_dp_result}, time = {three_dp_time:.8f} seconds")
print(f"Branch and Bound    -> {three_bb_result}, time = {three_bb_time:.8f} seconds")

=== THREE-PRODUCT RESULTS ===
Naive               -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00017958 seconds
Backtracking        -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00025042 seconds
Optimized           -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00002875 seconds
Dynamic Programming -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00013858 seconds
Branch and Bound    -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.02198408 seconds


In [19]:
print("=== FINAL COMPARISON ===")
print()

print("TWO PRODUCTS")
print(f"Naive               -> {two_naive_result}, time = {two_naive_time:.8f} seconds")
print(f"Backtracking        -> {two_backtracking_result}, time = {two_backtracking_time:.8f} seconds")
print(f"Optimized           -> {two_optimized_result}, time = {two_optimized_time:.8f} seconds")
print(f"Dynamic Programming -> {two_dp_result}, time = {two_dp_time:.8f} seconds")
print(f"Branch and Bound    -> {two_bb_result}, time = {two_bb_time:.8f} seconds")
print()

print("THREE PRODUCTS")
print(f"Naive               -> {three_naive_result}, time = {three_naive_time:.8f} seconds")
print(f"Backtracking        -> {three_backtracking_result}, time = {three_backtracking_time:.8f} seconds")
print(f"Optimized           -> {three_optimized_result}, time = {three_optimized_time:.8f} seconds")
print(f"Dynamic Programming -> {three_dp_result}, time = {three_dp_time:.8f} seconds")
print(f"Branch and Bound    -> {three_bb_result}, time = {three_bb_time:.8f} seconds")

=== FINAL COMPARISON ===

TWO PRODUCTS
Naive               -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00003125 seconds
Backtracking        -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00004637 seconds
Optimized           -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00000671 seconds
Dynamic Programming -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00004271 seconds
Branch and Bound    -> {'rockers': 2, 'stools': 6, 'profit': 230}, time = 0.00061012 seconds

THREE PRODUCTS
Naive               -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00017958 seconds
Backtracking        -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00025042 seconds
Optimized           -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00002875 seconds
Dynamic Programming -> {'rockers': 0, 'stools': 1, 'new_items': 7, 'profit': 330}, time = 0.00013858 seconds
Branch and Bound    -> {'rockers': 0, 'stool